# From Scratch GRPO: Step by Step

In [ ]:
import re
def extract_boxed_answer(text:str) -> str | None:
  result = re.search(r"\\boxed\{([^}]*)\}",text)
  if result:
    return result.group(1).strip()
  return None
# Test
text = "the answer is: \\boxed{42}"
extract_boxed_answer(text)

'42'

In [ ]:
def grad_answer(predicted:str, ground_truth:str) -> bool:
  predicted = predicted.strip().replace(" ","")
  ground_truth = ground_truth.strip().replace(" ","")
  if predicted == ground_truth:
    return True
  try:
    return abs(float(predicted)-float(ground_truth)) < 1e-6
  except:
    return False
# Test
assert grad_answer("23","32")==False, "Uncomplete"
assert grad_answer("23","23")==True, "Unomplete"

In [ ]:
def reward_rlvr(response:str,ground_truth:str) ->  float:
  boxed_response = extract_boxed_answer(response)
  if boxed_response:
    grad = grad_answer(boxed_response,ground_truth)
    return float(grad)
  return 0.0
# Test
response = "the answer is: \\boxed{42}"
ground_truth = "42"
reward_rlvr(response,ground_truth)

1.0

In [ ]:
# compute grpo loss
# 1. load model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_id = "LiquidAI/LFM2-350M"
device = "auto"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/91.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

In [ ]:
# test
prompt = "hello"
input_ids = tokenizer.encode(prompt,return_tensors="pt",device=model.device)
output_ids = model.generate(
      input_ids,
      max_new_tokens = 32,
      temperature = 0.8,
      do_sample = True
  )

output_ids

tensor([[    1, 52572, 22583,  1132,  1249,  1236,  2538,   906, 22583,  1132,
           792,   969,   770,   843, 28794,   875,  2979,   579, 16578, 22583,
          1132,   995,  1109,   856,   768,   730,  1600,   533,  5458, 16459,
          3316, 10879,   968, 15026]])

In [ ]:
tokenizer.decode(output_ids.squeeze(0))

'<|startoftext|>hello carnations – also known as carnations fieser (French for ‘fairy carnations’), is a 1988 German drama film directed by Friedrich'

In [ ]:
# compute grpo loss
# 1. generate rollout
def generate_rollout(model,
                     tokenizer,
                     prompt:str,
                     num_rollout:int=4,
                     max_new_tokens=512,
                     temperature=0.8):
  input_ids = tokenizer.encode(prompt,return_tensors="pt",device=model.device)
  rollout = []
  rollout_output_ids = []
  for r in range(num_rollout):
    output_ids = model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        temperature = temperature,
        do_sample = True
    ).squeeze(0).tolist()
    response = tokenizer.decode(output_ids[input_ids.shape[-1]:],skip_special_tokens=True)
    rollout.append(response)
    rollout_output_ids.append(output_ids)
  return {"input_ids_shape":input_ids.shape[-1],"rollout_output_ids":rollout_output_ids,"rollout":rollout}


In [ ]:
# test
from pprint import pprint
result = generate_rollout(model,
                     tokenizer,
                     prompt,
                     num_rollout=2,
                     max_new_tokens=13,
                     temperature=0.8)
pprint(result)

{'input_ids_shape': 2,
 'rollout': [' einem ist ein 1995 von der Arsis Films GmbH und',
             'animals is a YouTube channel which focuses on videos about '
             'animals.'],
 'rollout_output_ids': [[1,
                         52572,
                         4682,
                         2168,
                         1503,
                         730,
                         1344,
                         530,
                         1661,
                         1140,
                         1508,
                         14110,
                         31401,
                         24071,
                         1044],
                        [1,
                         52572,
                         19745,
                         1486,
                         856,
                         768,
                         18394,
                         10206,
                         1144,
                         16317,
                         884,
      

In [ ]:
rewards = []
for r in result["rollout"]:
  reward = reward_rlvr(r,"42")
  rewards.append(reward)
result["rewards"]=rewards
pprint(result)

{'input_ids_shape': 2,
 'rewards': [0.0, 0.0],
 'rollout': [' einem ist ein 1995 von der Arsis Films GmbH und',
             'animals is a YouTube channel which focuses on videos about '
             'animals.'],
 'rollout_output_ids': [[1,
                         52572,
                         4682,
                         2168,
                         1503,
                         730,
                         1344,
                         530,
                         1661,
                         1140,
                         1508,
                         14110,
                         31401,
                         24071,
                         1044],
                        [1,
                         52572,
                         19745,
                         1486,
                         856,
                         768,
                         18394,
                         10206,
                         1144,
                         16317,
            

In [ ]:
# calculate adventages
def calculate_advantage(rewards,epsilon=1e-8,device="cpu"):
  rewards = torch.tensor(rewards,device=device)
  rewards_mean = rewards.mean()
  rewards_std = rewards.std()
  advantages = (rewards-rewards_mean)/(rewards_std+epsilon)
  return advantages
# test
print(calculate_advantage([1.0,-1.0]))
print(calculate_advantage([1.0,1.0]))

tensor([ 0.7071, -0.7071])
tensor([0., 0.])


In [ ]:
# calculate loss
def log_probas(model,input_ids,prompt_length,device="cpu"):
  input_ids = torch.tensor(input_ids,device=device).unsqueeze(0)
  logits = model(input_ids).logits.squeeze(0)
  output_target_logits = logits[:-1,:]
  target_ids = input_ids.squeeze(0).tolist()[1:]
  log_probs = torch.log_softmax(logits,dim=-1)
  target_ids = input_ids[:,1:].squeeze(0)
  target_log_probs = log_probs.gather(
      -1,
      index = target_ids.unsqueeze(1) # [seq,1]
  ).squeeze(1)
  return target_log_probs[prompt_length-1:].sum()

In [ ]:
#test
input_ids = result["rollout_output_ids"][0]
prompt_length = result["input_ids_shape"]
prompt_length
log_prob = log_probas(model,input_ids,prompt_length)
log_prob

-44.0

In [ ]:
def grpo_loss(
    model,
    tokenizer,
    prompt:str,
    ground_truth,
    num_rollout:int=4,
    max_new_tokens:int=512,
    temperature:float=0.8,
    device:str="cpu") -> dict :
    result = generate_rollout(model,
                     tokenizer,
                     prompt,
                     num_rollout=num_rollout,
                     max_new_tokens=max_new_tokens,
                     temperature=temperature) #input_ids_shape,rollout_output_ids,rollout
    rewards = []
    for r in result["rollout"]:
      reward = reward_rlvr(r,ground_truth)
      rewards.append(reward)
    result["rewards"]=rewards
    advantages = calculate_advantage(result["rewards"],device=device)
    if torch.allclose(advantages,torch.zeros_like(advantages),atol=1e-8):
      return {"loss":0.0, "loss_tensor":None, "rewards":result["rewards"]}
    log_prob_list = []
    for oi in result["rollout_output_ids"]:
      log_prob = log_probas(model,oi,result["input_ids_shape"],device=device)
      log_prob_list.append(log_prob)

    logps = torch.stack(log_prob_list)
    loss = - (advantages.detach() *  logps).mean()
    return {"loss":loss.item(),"loss_tensor":loss, "rewards":result["rewards"], "advantages":advantages.tolist()}

In [ ]:
# test
grpo_loss(
    model,
    tokenizer,
    "hello what is the result of 31+11 put the final answer in this format \\boxed{answer}",
    "42",
    num_rollout=3,
    max_new_tokens=256,
    temperature=0.8,
    device="cpu")

{'loss': 2.3875837326049805,
 'loss_tensor': tensor(2.3876, grad_fn=<NegBackward0>),
 'rewards': [1.0, 0.0, 1.0],
 'advantages': [0.5773502588272095, -1.1547006368637085, 0.5773502588272095]}

In [ ]:
def train_rlvr(
    model,
    tokenizer,
    train_data:list[dict],
    steps=100,
    num_rollout=4,
    temperature=0.8,
    max_new_token=512,
    lr=1e-5,
    device="cpu"
):
  optimizer = torch.optim.AdamW(model.parameters(),lr=lr)
  model.train()
  for step in range(steps):
    input = train_data[step%len(train_data)]
    prompt = (
            f"Solve the following problem. Put your final answer within "
            f"\\boxed{{}}.\n\nProblem: {input['problem']}"
        )
    answer = input["answer"]
    stats = grpo_loss(
      model,
      tokenizer,
      prompt,
      answer,
      num_rollout=num_rollout,
      max_new_tokens=max_new_token,
      temperature=temperature,
      device=device)
    if stats["loss_tensor"] is not None:
      optimizer.zero_grad()
      stats["loss_tensor"].backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      optimizer.step()
    reward_avg = sum(stats["rewards"]) / len(stats["rewards"])
    if (step + 1) % 5 == 0:
      print(f"Step {step+1:3d} | loss={stats['loss']:.4f} | "
            f"reward_avg={reward_avg:.3f}")
  return model

In [ ]:
model = train_rlvr(
    model,
    tokenizer,
    train_data = [
        {"problem":"what is the result of 31+11","answer":"42"},
        {"problem":"what is the value of pi","answer":"3.14"},
        {"problem":"what is the value of x in equation 2x+3=0","answer":"-3/2"},
        ],
    steps=10,
    num_rollout=4,
    temperature=0.8,
    max_new_token=64,
    lr=1e-5,
    device="cpu"
)

Step   5 | loss=0.0000 | reward_avg=0.000
Step  10 | loss=-9.8379 | reward_avg=0.750


# GRPO Real Application

Suppose the entreprise system exposes only three financial APIs:
1. `get_stock_price` : required args `ticker`, `date`
2. `get_revenue` : required args `company`, `fiscal_year`
3. `convert_currency` : `amount`, `from_currency`, `to_currency`

After the model generates JSOM, we check whether it can be parsed, wheter the function name exists, wether the arguments match the schema, and whether execution  returns the expected result.

user_request -> SLM -> condidate tool-call JSON -> Verifier (Json, Schema, Execution) -> reward -> GRPO Update

### Load Model

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
device = "mps"
model = AutoModelForCausalLM.from_pretrained(model_id,device_map=device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/Users/aelmajjodi/Desktop/time_scaling_techniques/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 407.18it/s]


### Dataset Preparation

In [2]:
TOOLS = [
    {
        "name": "get_stock_price",
        "description": "Get the closing stock price for a ticker on a date.",
        "parameters": {
            "ticker": {"type": "string"},
            "date": {"type": "string", "format": "YYYY-MM-DD"},
        },
        "required": ["ticker", "date"],
    },
    {
        "name": "get_revenue",
        "description": "Get annual revenue for a company and fiscal year.",
        "parameters": {
            "company": {"type": "string"},
            "fiscal_year": {"type": "integer"},
        },
        "required": ["company", "fiscal_year"],
    },
    {
        "name": "convert_currency",
        "description": "Convert money between currencies.",
        "parameters": {
            "amount": {"type": "number"},
            "from_currency": {"type": "string"},
            "to_currency": {"type": "string"},
        },
        "required": ["amount", "from_currency", "to_currency"],
    }
]

In [3]:
STOCK_DB = {
    ("AAPL", "2025-01-02"): 243.85,
    ("AAPL", "2025-01-03"): 243.36,
    ("MSFT", "2025-01-02"): 418.58,
    ("MSFT", "2025-01-03"): 423.35,
}

REVENUE_DB = {
    ("Apple", 2024): 391_035_000_000,
    ("Microsoft", 2024): 245_122_000_000,
    ("Tesla", 2024): 97_690_000_000,
}

FX_DB = {
    ("USD", "EUR"): 0.92,
    ("EUR", "USD"): 1.09,
    ("USD", "JPY"): 157.2,
}


In [4]:
def execute_tool(tool_name:str, arguments:dict):
  match tool_name:
    case "get_stock_price":
      return STOCK_DB[(arguments['ticker'],arguments['date'])]
    case "get_revenue":
      return REVENUE_DB[(arguments['company'],arguments['fiscal_year'])]
    case "convert_currency":
      return round(FX_DB[(arguments['from_currency'],arguments['to_currency'])]*arguments["amount"],2)
    case _:
      return ValueError(f"Unkown tool: {tool_name}")

In [5]:
execute_tool("get_stock_price",{'ticker':'AAPL','date':'2025-01-03'})

243.36

In [6]:
import json
def format_prompt(user_query:str,tokenizer) -> str:
  refined_prompt = (
      "You are a helpful financial assistant. Choose exactly one tool call"
      "if a tool needed. Return only JSON with this shape: "
      '{"name":"...", "arguments": {...}}. '
      "If no tool is needed, return "
      '{"name":"no_call", "arguments": {}}.\n\n'
      f"Availabe tools:\n{json.dumps(TOOLS, ensure_ascii=False, indent=2)}\n\n"
  )
  messages = [
      {"role":"system","content":refined_prompt},
      {"role":"user","content":user_query},
  ]
  refined_prompt = tokenizer.apply_chat_template(messages,add_generation_prompt=True,tokenize=False)
  return refined_prompt

In [7]:
format_prompt("hello",tokenizer)

'<|im_start|>system\nYou are a helpful financial assistant. Choose exactly one tool callif a tool needed. Return only JSON with this shape: {"name":"...", "arguments": {...}}. If no tool is needed, return {"name":"no_call", "arguments": {}}.\n\nAvailabe tools:\n[\n  {\n    "name": "get_stock_price",\n    "description": "Get the closing stock price for a ticker on a date.",\n    "parameters": {\n      "ticker": {\n        "type": "string"\n      },\n      "date": {\n        "type": "string",\n        "format": "YYYY-MM-DD"\n      }\n    },\n    "required": [\n      "ticker",\n      "date"\n    ]\n  },\n  {\n    "name": "get_revenue",\n    "description": "Get annual revenue for a company and fiscal year.",\n    "parameters": {\n      "company": {\n        "type": "string"\n      },\n      "fiscal_year": {\n        "type": "integer"\n      }\n    },\n    "required": [\n      "company",\n      "fiscal_year"\n    ]\n  },\n  {\n    "name": "convert_currency",\n    "description": "Convert mon

In [8]:
# create a simple dataset
from datasets import Dataset
def make_dataset() -> Dataset:
  rows = []
  examples = [
      (
          "What was AAPL's closing price on 2025-01-03?",
          {"name": "get_stock_price", "arguments": {"ticker": "AAPL", "date": "2025-01-03"}},
      ),
      (
          "Get MSFT close price on 2025-01-02.",
          {"name": "get_stock_price", "arguments": {"ticker": "MSFT", "date": "2025-01-02"}},
      ),
      (
          "How much revenue did Tesla report in fiscal year 2024?",
          {"name": "get_revenue", "arguments": {"company": "Tesla", "fiscal_year": 2024}},
      ),
      (
          "Convert 120 USD to EUR.",
          {
              "name": "convert_currency",
              "arguments": {"amount": 120, "from_currency": "USD", "to_currency": "EUR"},
          },
      ),
  ]

  for query, gold_call in examples:
    result = execute_tool(gold_call["name"],gold_call["arguments"])
    rows.append(
        {
            "prompt": format_prompt(query,tokenizer),
            "gold_call": gold_call,
            "expected_result": str(result)
        }
    )
  return Dataset.from_list(rows)

In [9]:
dataset = make_dataset()
dataset

Dataset({
    features: ['prompt', 'gold_call', 'expected_result'],
    num_rows: 4
})



> A real experiment should expand each tool into dozens or hundreds of query templates and include negative cases: missing arguments, wrong date formats, no available tool, and confusingly similar tools.



### Reward Function

In [10]:
import re
def extract_answer(completion:str)->str:
  match = re.search(r'</think>\s*(\{[\s\S]*?\})\s*<\|im_end\|>', completion)
  if not match:
    return completion
  return match.group(1)

In [28]:
# 1. json parser
def parse_call(text:str)->dict|None:
  text = extract_answer(text)
  try:
    return json.loads(text.strip())
  except json.JSONDecodeError:
    return None
# test
print(parse_call('[1,2,3]'))

[1, 2, 3]


In [29]:
# schema correctness
def schema_ok(call:dict, gold:dict) -> bool:
  if call.get("name") != gold.get("name"):
    return False
  if not isinstance(call.get("arguments"), dict):
    return False

  gold_args = gold.get("arguments", {})
  call_args = call.get("arguments", {})

  for key, value in gold_args.items():
    if key not in call_args:
      return False
    # Basic type checking
    if type(value) != type(call_args[key]):
      return False
  return True

In [30]:
# reward function
def tool_reward(completions, gold_call, expected_result, **kwargs):
  rewards = []
  # In GRPOTrainer, completions is a list, and gold_call/expected_result are lists of the same length
  for c, gc, er in zip(completions, gold_call, expected_result):
    reward = 0.0
    c_parsed = parse_call(c)
    if c_parsed is None:
      rewards.append(0.0)
      continue
    if not isinstance(c_parsed,dict):
      rewards.append(0.0)
      continue
    # 1. Format reward (JSON was parseable)
    reward += 0.2

    # 2. Name correctness
    if c_parsed.get("name") == gc.get("name"):
      reward += 0.3

    # 3. Schema correctness
    if schema_ok(c_parsed, gc):
      reward += 0.3

    # 4. Execution correctness
    if gc.get("name") == "no_call":
      if c_parsed.get("name") == "no_call":
        reward += 0.2
    else:
      try:
        # Use c_parsed here, not c
        result = execute_tool(c_parsed["name"], c_parsed["arguments"])
        if str(result) == str(er):
          reward += 0.2
      except Exception:
        pass

    rewards.append(reward)
  return rewards

### Test Model

In [14]:
prompt = "What was AAPL's closing price on 2025-01-03?"
formated_prompt = format_prompt(prompt,tokenizer)
formated_prompt

'<|im_start|>system\nYou are a helpful financial assistant. Choose exactly one tool callif a tool needed. Return only JSON with this shape: {"name":"...", "arguments": {...}}. If no tool is needed, return {"name":"no_call", "arguments": {}}.\n\nAvailabe tools:\n[\n  {\n    "name": "get_stock_price",\n    "description": "Get the closing stock price for a ticker on a date.",\n    "parameters": {\n      "ticker": {\n        "type": "string"\n      },\n      "date": {\n        "type": "string",\n        "format": "YYYY-MM-DD"\n      }\n    },\n    "required": [\n      "ticker",\n      "date"\n    ]\n  },\n  {\n    "name": "get_revenue",\n    "description": "Get annual revenue for a company and fiscal year.",\n    "parameters": {\n      "company": {\n        "type": "string"\n      },\n      "fiscal_year": {\n        "type": "integer"\n      }\n    },\n    "required": [\n      "company",\n      "fiscal_year"\n    ]\n  },\n  {\n    "name": "convert_currency",\n    "description": "Convert mon

In [15]:
ids = tokenizer.encode(formated_prompt,return_tensors="pt").to(device)
resp_ids = model.generate(ids,max_new_tokens=256)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [16]:
completion = tokenizer.batch_decode(resp_ids)[0]
print(completion[len(formated_prompt):])

The closing price for Apple Inc. (AAPL) on January 3, 2025, is 1,464.25 USD.<|im_end|>


In [17]:
print(parse_call(completion))

None


### Train

In [18]:
from peft import LoraConfig
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    task_type = "CAUSAL_LM"
)

In [19]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    output_dir="./output",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    # max_prompt_length=2048,
    max_completion_length=256,
    temperature=0.8,
    logging_steps=1,
    save_steps=50,
    max_steps=100,
    report_to="wandb",
    run_name="smolln_api_tool_call_v0.3",
)

In [31]:
trainer = GRPOTrainer(
    model=model,#"Qwen/Qwen3-0.6B",
    args=training_args,
    train_dataset=dataset,
    reward_funcs=tool_reward,
    peft_config=lora_config
)

/Users/aelmajjodi/Desktop/time_scaling_techniques/.venv/lib/python3.13/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/aelmajjodi/Desktop/time_scaling_techniques/.venv/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
trainer.train()

/Users/aelmajjodi/Desktop/time_scaling_techniques/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.000000
2,-0.017221
3,0.000000
4,-0.284707
5,-0.069090
6,0.000000
7,0.063105
8,0.000000
9,0.000000
10,-0.142583


2026-06-27 16:14:21.590 Python[91233:9887976] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-91233-2026-06-27_16_14_21-17032415‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-06-27 16:14:24.792 Python[91233:9887976] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-91233-2026-06-27_16_14_24-1772467954‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-06-27 16:14:24.906 Python[91233:9887976] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-91233-2026-06-27_16_14_24-559864734‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-06-27 16:14:25.070 Python[91233:9887976] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-91233-2026-06-27_16_14_25-734899240‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of

InterruptedError: [Errno 4] Interrupted system call

wandb-core(10734) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(12164) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(13592) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(14716) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(15265) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(15847) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(16344) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(16920) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(17486) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(18040) MallocStackLogging: can't turn off malloc stack logging because 

# Tests

In [22]:
# Test the reward function with a mock successful completion
test_sample = dataset[0]
mock_completion = "<|im_start|>assistant\n<think>I need to find the stock price for AAPL on 2025-01-03.</think>\n{\"name\": \"get_stock_price\", \"arguments\": {\"ticker\": \"AAPL\", \"date\": \"2025-01-03\"}}\n<|im_end|>"

# tool_reward expects lists for completions, gold_call, and expected_result
test_rewards = tool_reward(
    completions=[mock_completion],
    gold_call=[test_sample['gold_call']],
    expected_result=[test_sample['expected_result']]
)

print(f"Prompt: {test_sample['prompt'][:100]}...")
print(f"Gold Call: {test_sample['gold_call']}")
print(f"Mock Reward Output: {test_rewards}")

assert test_rewards[0] > 0, "Reward is still zero! Check the parsing logic or extraction regex."

Prompt: <|im_start|>system
You are a helpful financial assistant. Choose exactly one tool callif a tool need...
Gold Call: {'name': 'get_stock_price', 'arguments': {'ticker': 'AAPL', 'date': '2025-01-03', 'company': None, 'fiscal_year': None, 'amount': None, 'from_currency': None, 'to_currency': None}}
Mock Reward Output: [0.7]


In [29]:
# Debug: Check what the model is actually generating right now
# If all 4 outputs look the same and are wrong, loss will stay 0.0

test_prompt = dataset[0]['prompt']
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

print("--- MODEL GENERATIONS (CHECKING FOR DIVERSITY) ---")
for i in range(4):
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"Generation {i+1}:\n{text[len(test_prompt):]}\n")
    print(f"Reward: {tool_reward([text], [dataset[0]['gold_call']], [dataset[0]['expected_result']])[0]}")
    print("-" * 30)

--- MODEL GENERATIONS (CHECKING FOR DIVERSITY) ---
Generation 1:
<think>
Okay, the user is asking for AAPL's closing price on January 3rd, 2025. Let me check which tool I can use. The available tools are get_stock_price, get_revenue, and convert_currency.

The get_stock_price tool requires a ticker and a date. The user provided both: ticker is AAPL and date is 2025-01-03. I need to make sure the parameters are correctly formatted. The date should be in the format YYYY-MM-DD, which it is. So I should call get_stock_price with ticker AAPL and date 2025-01-03. The other tools don't apply here since the question is about a stock price, not revenue or currency conversion. No need for other tools. So the correct JSON call is to get_stock_price with those parameters.
</think>

{"name": "get_stock_price", "arguments": {"ticker": "AAPL", "date": "2025-01-03"}}<|im_end|>

Reward: 0.7
------------------------------
Generation 2:
<think>
Okay, the user is asking for AAPL's closing price on January

In [33]:
import torch
torch.mps.empty_cache()

wandb-core(35697) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35703) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35708) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35710) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35713) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35715) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35723) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35725) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35728) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(35731) MallocStackLogging: can't turn off malloc stack logging because 